# Data-Driven Causal Analysis with DoWhy

This notebook demonstrates **statistical causal inference** using the
[DoWhy](https://www.pywhy.org/dowhy/) library on synthetic datasets generated
by our `causal_inference.data` module.

While Notebooks 01 and 02 focus on the *conceptual* and *LLM-driven* aspects
of causal inference, this notebook shows the complementary **data-driven** side:

1. **Specify a causal model** (DAG) programmatically
2. **Identify the causal estimand** using the DAG
3. **Estimate the causal effect** with multiple estimators
4. **Refute the estimate** with robustness checks
5. **Compare naive vs. causal estimates** to illustrate confounding bias

This represents the "statistical backbone" that the L2 Intervention Agent
in the agentic workflow would invoke when real data is available.

In [ ]:
# Optional — uncomment to install dependencies
# %pip install dowhy econml networkx matplotlib numpy pandas

In [ ]:
import sys
from pathlib import Path

src_dir = Path.cwd().parent
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from causal_inference.data.synthetic import (
    generate_confounded_data,
    generate_rct_data,
    generate_iv_data,
)
from causal_inference.viz.dag import draw_dag_from_spec

plt.style.use("seaborn-v0_8-whitegrid")
print("Setup complete.")

---

## 1. Confounded Scenario: Naive vs. Causal Estimation

We generate data where **Genetics** confounds the relationship between
**Smoking** and **Cancer**. A naive regression gives a biased estimate;
DoWhy's back-door adjustment recovers the true ATE.

In [ ]:
ds = generate_confounded_data(n=5000, treatment_effect=0.3)
df = ds.data

fig = draw_dag_from_spec(ds.dag, figsize=(7, 4),
                         title="Confounded: Genetics → Smoking → Cancer, Genetics → Cancer")
plt.show()

print(f"True ATE: {ds.true_ate}")
print(f"\nDataset shape: {df.shape}")
df.describe().round(3)

In [ ]:
import dowhy
from dowhy import CausalModel

# DoWhy uses a GML string to specify the causal graph
gml_graph = """
graph [
    directed 1
    node [ id "genetics" label "genetics" ]
    node [ id "smoking" label "smoking" ]
    node [ id "cancer_score" label "cancer_score" ]
    edge [ source "genetics" target "smoking" ]
    edge [ source "genetics" target "cancer_score" ]
    edge [ source "smoking" target "cancer_score" ]
]
"""

model = CausalModel(
    data=df,
    treatment="smoking",
    outcome="cancer_score",
    graph=gml_graph,
)

model.view_model()

### Step 1: Identify the Causal Estimand

DoWhy automatically finds the appropriate identification strategy
(back-door criterion here) and expresses the estimand.

In [ ]:
estimand = model.identify_effect(proceed_when_unidentifiable=True)
print(estimand)

### Step 2: Estimate the Causal Effect

We compare multiple estimators to show convergence to the true ATE.

In [ ]:
from scipy import stats as sp_stats

# Naive estimate (no adjustment)
naive_slope = sp_stats.linregress(df["smoking"], df["cancer_score"]).slope

# DoWhy: Linear regression with back-door adjustment
estimate_lr = model.estimate_effect(
    estimand,
    method_name="backdoor.linear_regression",
)

# DoWhy: Propensity Score Weighting (IPW)
estimate_ipw = model.estimate_effect(
    estimand,
    method_name="backdoor.propensity_score_weighting",
    method_params={"weighting_scheme": "ips_weight"},
)

print("═" * 60)
print("ESTIMATION RESULTS")
print("═" * 60)
print(f"\n  True ATE:                           {ds.true_ate:.4f}")
print(f"  Naive regression (no adjustment):    {naive_slope:.4f}  (biased!)")
print(f"  Back-door (linear regression):       {estimate_lr.value:.4f}")
print(f"  Back-door (IPW):                     {estimate_ipw.value:.4f}")

### Step 3: Refute the Estimate

DoWhy provides automated refutation tests to check the robustness of causal claims.

In [ ]:
# Refutation 1: Add a random common cause (placebo confounder)
refute_random = model.refute_estimate(
    estimand, estimate_lr,
    method_name="random_common_cause",
)
print(refute_random)
print()

In [ ]:
# Refutation 2: Placebo treatment (replace treatment with random noise)
refute_placebo = model.refute_estimate(
    estimand, estimate_lr,
    method_name="placebo_treatment_refuter",
    placebo_type="permute",
)
print(refute_placebo)

In [ ]:
# Refutation 3: Data subset refuter (stability across subsamples)
refute_subset = model.refute_estimate(
    estimand, estimate_lr,
    method_name="data_subset_refuter",
    subset_fraction=0.8,
)
print(refute_subset)

In [ ]:
# Summary visualization
fig, ax = plt.subplots(figsize=(8, 5))

methods = ["True ATE", "Naive\n(no adjustment)", "Back-door\n(OLS)", "Back-door\n(IPW)"]
estimates = [ds.true_ate, naive_slope, estimate_lr.value, estimate_ipw.value]
colors = ["#0984e3", "#e17055", "#00b894", "#00b894"]

bars = ax.bar(methods, estimates, color=colors, edgecolor="#2d3436", linewidth=1.2)
ax.axhline(y=ds.true_ate, color="#0984e3", linestyle="--", alpha=0.7, label=f"True ATE = {ds.true_ate}")

for bar, val in zip(bars, estimates):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f"{val:.4f}", ha="center", fontweight="bold", fontsize=11)

ax.set_ylabel("Estimated Effect", fontsize=12)
ax.set_title("Naive vs. Causal Estimates (Confounded Scenario)", fontsize=13, fontweight="bold")
ax.legend(fontsize=11)
fig.tight_layout()
plt.show()

---

## 2. RCT Scenario: When Naive Estimation Works

In a randomized controlled trial (no confounders), the naive difference-in-means
is already an unbiased estimate of the ATE. DoWhy confirms this.

In [ ]:
ds_rct = generate_rct_data(n=3000, true_effect=0.6)
df_rct = ds_rct.data

fig = draw_dag_from_spec(ds_rct.dag, figsize=(5, 3), title="RCT: Treatment → Recovery")
plt.show()

# Naive difference in means
treated = df_rct[df_rct["treatment"] == 1]["recovery"]
control = df_rct[df_rct["treatment"] == 0]["recovery"]
naive_diff = treated.mean() - control.mean()

print(f"True ATE:                  {ds_rct.true_ate:.4f}")
print(f"Naive difference-in-means: {naive_diff:.4f}")
print(f"\n✓ In an RCT, naive estimation is unbiased.")

In [ ]:
gml_rct = """
graph [
    directed 1
    node [ id "treatment" label "treatment" ]
    node [ id "recovery" label "recovery" ]
    edge [ source "treatment" target "recovery" ]
]
"""

model_rct = CausalModel(
    data=df_rct,
    treatment="treatment",
    outcome="recovery",
    graph=gml_rct,
)

estimand_rct = model_rct.identify_effect(proceed_when_unidentifiable=True)
estimate_rct = model_rct.estimate_effect(
    estimand_rct,
    method_name="backdoor.linear_regression",
)

print(f"DoWhy estimate: {estimate_rct.value:.4f}")
print(f"True ATE:       {ds_rct.true_ate:.4f}")

---

## 3. Instrumental Variable Scenario

When the confounder is **unmeasured** and no front-door path exists,
an **instrumental variable** (IV) can identify the causal effect.

We generate data with:
- **Z** (instrument): affects Treatment but not Outcome directly
- **U** (unmeasured confounder): affects both Treatment and Outcome

OLS is biased; 2SLS (IV estimation) recovers the true effect.

In [ ]:
ds_iv = generate_iv_data(n=5000, treatment_effect=0.4)
df_iv = ds_iv.data

fig = draw_dag_from_spec(ds_iv.dag, figsize=(8, 4),
                         title="IV Scenario: Z → X → Y, U → X, U → Y")
plt.show()

print(f"True ATE: {ds_iv.true_ate}")
print(f"Dataset: {df_iv.shape}")
print(f"\nNote: 'instrument' (Z) is observed; the confounder (U) is NOT in the dataset.")
df_iv.head()

In [ ]:
from scipy import stats as sp_stats

# Naive OLS (biased due to unmeasured confounding)
naive_ols = sp_stats.linregress(df_iv["treatment"], df_iv["outcome"]).slope

# Manual 2SLS (Two-Stage Least Squares)
# Stage 1: regress treatment on instrument
stage1 = sp_stats.linregress(df_iv["instrument"], df_iv["treatment"])
treatment_hat = stage1.slope * df_iv["instrument"] + stage1.intercept

# Stage 2: regress outcome on predicted treatment
stage2 = sp_stats.linregress(treatment_hat, df_iv["outcome"])
iv_estimate = stage2.slope

# Wald estimator (simple IV formula)
cov_zy = np.cov(df_iv["instrument"], df_iv["outcome"])[0, 1]
cov_zx = np.cov(df_iv["instrument"], df_iv["treatment"])[0, 1]
wald_estimate = cov_zy / cov_zx

print("═" * 60)
print("INSTRUMENTAL VARIABLE ESTIMATION")
print("═" * 60)
print(f"\n  True ATE:              {ds_iv.true_ate:.4f}")
print(f"  Naive OLS (biased):    {naive_ols:.4f}  ← inflated by confounding")
print(f"  2SLS estimate:         {iv_estimate:.4f}")
print(f"  Wald estimate:         {wald_estimate:.4f}")
print(f"\n✓ IV estimation recovers the true causal effect")
print(f"  despite the unmeasured confounder.")

In [ ]:
# DoWhy IV estimation
gml_iv = """
graph [
    directed 1
    node [ id "instrument" label "instrument" ]
    node [ id "treatment" label "treatment" ]
    node [ id "outcome" label "outcome" ]
    node [ id "Unobserved" label "Unobserved" ]
    edge [ source "instrument" target "treatment" ]
    edge [ source "treatment" target "outcome" ]
    edge [ source "Unobserved" target "treatment" ]
    edge [ source "Unobserved" target "outcome" ]
]
"""

model_iv = CausalModel(
    data=df_iv,
    treatment="treatment",
    outcome="outcome",
    graph=gml_iv,
)

estimand_iv = model_iv.identify_effect(proceed_when_unidentifiable=True)
print(estimand_iv)

In [ ]:
estimate_iv_dowhy = model_iv.estimate_effect(
    estimand_iv,
    method_name="iv.instrumental_variable",
    method_params={"iv_instrument_name": "instrument"},
)

print(f"DoWhy IV estimate: {estimate_iv_dowhy.value:.4f}")
print(f"True ATE:          {ds_iv.true_ate:.4f}")

---

## 4. Summary: Connecting Statistical and Agentic Approaches

| Scenario | Naive Estimate | Causal Estimate | Method | True ATE |
|----------|---------------|-----------------|--------|----------|
| **Confounded** | Biased (inflated) | Unbiased | Back-door adjustment | 0.30 |
| **RCT** | Unbiased | Unbiased | Difference-in-means | 0.60 |
| **IV** | Biased (inflated) | Unbiased | 2SLS / Wald | 0.40 |

### How This Connects to the Agentic Workflow

In the agentic workflow (Notebook 02), the **L2 Intervention Agent**:

1. Receives the DAG from the Orchestrator
2. Checks **identifiability** (back-door? front-door? IV?)
3. Selects the appropriate **estimator** (OLS, IPW, 2SLS, etc.)
4. Reports the result with **caveats**

This notebook shows what those estimators *actually do* with real data.
In a production system, the agentic workflow would invoke DoWhy or EconML
as tools, combining LLM reasoning about the DAG structure with
statistical estimation on the data.

```
┌─────────────────────┐     ┌─────────────────────┐
│  Agentic Workflow    │     │  Statistical Engine  │
│  (LLM + LangGraph)  │────▶│  (DoWhy / EconML)    │
│                      │◀────│                      │
│  • DAG specification │     │  • Identification    │
│  • Question routing  │     │  • Estimation        │
│  • Report synthesis  │     │  • Refutation        │
└─────────────────────┘     └─────────────────────┘
```